# Interval logic tutorial

[`LOGIC.md`](../LOGIC.md) defines the language and [`CODE.md`](../CODE.md) defines this Python API. This notebook uses complete-trace checking; symbolic reachability is a separate state-set interface.

In [ ]:
import portion as P

from interval_logic import (
    Always, NaturalSignal, Proposition, Selection, Signal, Trace,
    denote, format_formula, parse_formula, satisfies, satisfying_times,
)

## Formulas and physical-time parameters

Operands precede the optional parameter. A bare duration is half-open, `d]` is closed, an interval is absolute, and `None` means omission.

In [ ]:
available = Proposition("available")
fault = Proposition("fault")
job = Signal("job")

requirement = (available & ~fault).always("3]")
assignment = job.equals(2).eventually(P.closedopen(4, 8))
parsed = parse_formula("always_3] (available implies not fault)")
print(requirement)
print(format_formula(parsed, unicode=False))

## Complete-trace checking

A `Trace` owns its represented universe, proposition valuation, and complete natural-valued signals. `satisfies` implements fundamental local truth and `satisfying_times` collects its satisfying anchors; `denote` additionally preserves a selector's full anchored mask.

In [ ]:
universe = P.closed(0, 10)
trace = Trace(
    universe,
    propositions={
        "available": P.closedopen(2, 8),
        "fault": P.closedopen(6, 7),
    },
    signals={
        "job": NaturalSignal({0: P.closedopen(0, 4), 2: P.closed(4, 10)})
    },
)

print(satisfying_times("eventually_2] available", trace))
print(satisfies(assignment, trace, 3))

## Selection is a mask, not a condition

Selection is a time mask while a condition is globally true or false at its anchor, but both use the same future-directed window. At anchor 9, the absolute window below is empty, so the selector uses its outside branch everywhere and the universal condition is vacuously true.

In [ ]:
selector = Selection(available, fault, P.closed(2, 7))
condition = Always(available, P.closed(2, 7))

print("future-truncated anchored selector:", denote(selector, trace, 9))
print("condition at 9 (empty future window):", satisfies(condition, trace, 9))

## Rewriting and reachability

Use `simplify`, `expand_derived`, and `to_nnf` for formula rewriting. For a scheduler or transition model, implement `ReachabilityBackend` with a multidimensional `StateSet` and call `reach`. `ReachResult.tlt_set` is the computed TLT set, and `certified` reports whether common-trajectory exactness has been established.